In [1]:
import json
from pathlib import Path

INPUT_DIR = Path("dataset")
OUTPUT_FILE = Path("dataset/dataset.jsonl")

count = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:
    for file in sorted(INPUT_DIR.glob("*.json")):
        with open(file, "r", encoding="utf-8") as infile:
            obj = json.load(infile)
            outfile.write(json.dumps(obj))
            outfile.write("\n")
            count += 1

print(f"Saved {count} examples to {OUTPUT_FILE}")

Saved 29 examples to dataset.jsonl


In [4]:
from openai import OpenAI
from pathlib import Path
import datetime
import dotenv
import json
import os
import random
import re


def generate_dataset(
    version: str,
    n: int,
    tools: list[dict],
    base_dir: str = "../datasets",
):
    dotenv.load_dotenv()

    client = OpenAI(
        api_key=os.getenv("OPENAI_API_KEY")
    )

    dataset_dir = Path(base_dir) / version
    prompt_dir = dataset_dir / "prompts"

    dataset_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    prompt_path = prompt_dir / "system_prompt.txt"
    tools_path = dataset_dir / "tools.json"
    output_file = dataset_dir / "train.jsonl"

    if not prompt_path.exists():
        raise FileNotFoundError(
            f"Missing system prompt: {prompt_path}"
        )

    system_prompt = prompt_path.read_text(
        encoding="utf-8"
    )

    # Automatically generate tools.json.
    tools_path.write_text(
        json.dumps(
            tools,
            indent=4,
            ensure_ascii=False,
        ) + "\n",
        encoding="utf-8",
    )

    tool_names = [
        tool["name"]
        for tool in tools
    ]

    print(f"Dataset: {version}")
    print(f"Output:  {output_file}")
    print(f"Tools:   {', '.join(tool_names)}")

    # Resume automatically.
    if output_file.exists():
        with open(
            output_file,
            "r",
            encoding="utf-8",
        ) as f:
            start = sum(
                1
                for line in f
                if line.strip()
            )
    else:
        start = 0

    print(f"Found {start} existing samples.")

    if start >= n:
        print("Dataset already complete.")
        return

    with open(
        output_file,
        "a",
        encoding="utf-8",
    ) as f:

        for i in range(start, n):

            tool = random.choice(tools)

            prompt = f""""""

            response = client.responses.create(
                model="gpt-4o",
                input=[
                    {
                        "role": "system",
                        "content": system_prompt,
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ],
            )

            text = response.output_text.strip()

            match = re.search(
                r"\{.*\}",
                text,
                flags=re.DOTALL,
            )

            if match:
                text = match.group(0)

            try:
                obj = json.loads(text)

                messages = obj["messages"]

                if len(messages) != 3:
                    raise ValueError(
                        "Expected exactly 3 messages."
                    )

                if messages[0]["role"] != "user":
                    raise ValueError(
                        "First message must be user."
                    )

                if messages[1]["role"] != "tool":
                    raise ValueError(
                        "Second message must be tool."
                    )

                if messages[2]["role"] != "assistant":
                    raise ValueError(
                        "Third message must be assistant."
                    )

                if not messages[2].get("content"):
                    raise ValueError(
                        "Assistant content is empty."
                    )

                if "tool_calls" in messages[2]:
                    raise ValueError(
                        "Summarizer must not produce tool_calls."
                    )

                f.write(
                    json.dumps(
                        obj,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

                f.flush()
                os.fsync(f.fileno())

                print(
                    f"{i + 1}/{n} ✓ {tool['name']}"
                )

            except (
                json.JSONDecodeError,
                KeyError,
                TypeError,
                ValueError,
            ) as e:

                print(
                    f"\n{i + 1}/{n} ✗ Invalid sample"
                )
                print(e)
                print("=" * 80)
                print(text)
                print("=" * 80)


In [7]:
TOOLS = [
        {
            "name": "gene_lookup",
            "description": "Retrieve information about a gene.",
            "arguments": [
                "gene_name"
            ],
        },
        {
            "name": "search_pubmed",
            "description": "Search PubMed literature.",
            "arguments": [
                "query"
            ],
        },
        {
            "name": "classify_sequence",
            "description": "Predict protein family.",
            "arguments": [
                "sequence"
            ],
        },
]

generate_dataset(
    version="summarizer-0.0.1",
    n=500,
    tools=TOOLS,
)

Dataset: summarizer-0.0.1
Output:  ../datasets/summarizer-0.0.1/train.jsonl
Tools:   gene_lookup, search_pubmed, classify_sequence
Found 10 existing samples.
11/500 ✓ gene_lookup
12/500 ✓ search_pubmed
13/500 ✓ classify_sequence
14/500 ✓ search_pubmed
15/500 ✓ classify_sequence
16/500 ✓ classify_sequence
17/500 ✓ search_pubmed
18/500 ✓ classify_sequence
19/500 ✓ gene_lookup
20/500 ✓ search_pubmed
21/500 ✓ classify_sequence
22/500 ✓ classify_sequence
23/500 ✓ search_pubmed
24/500 ✓ search_pubmed
25/500 ✓ gene_lookup
26/500 ✓ gene_lookup
27/500 ✓ classify_sequence
28/500 ✓ search_pubmed
29/500 ✓ search_pubmed
30/500 ✓ search_pubmed
31/500 ✓ search_pubmed
32/500 ✓ gene_lookup
33/500 ✓ gene_lookup
34/500 ✓ search_pubmed
35/500 ✓ gene_lookup
36/500 ✓ gene_lookup
37/500 ✓ classify_sequence
38/500 ✓ gene_lookup
39/500 ✓ classify_sequence
40/500 ✓ search_pubmed
41/500 ✓ search_pubmed
42/500 ✓ gene_lookup
43/500 ✓ gene_lookup
44/500 ✓ gene_lookup
45/500 ✓ search_pubmed
46/500 ✓ gene_lookup
47/5

In [1]:
#generate_dataset(500, "../datasets/v0.0.2/train.jsonl")

In [ ]:
baseline = {}

for p in prompts:

    ans = ask(p)

    baseline[p] = ans

    print("="*80)
    #print(p)
    print(ans)